# Практическая работа: Полиномиальная регрессия (Градиентный спуск + Автоподбор степени)

**Цель работы:** Реализовать класс полиномиальной регрессии. Добавить механизм автоматического поиска оптимальной степени полинома с помощью разделения выборки на обучающую (Train) и тестовую (Test).

## 1. Математическая постановка задачи
Полиномиальная регрессия — это расширение множественной линейной регрессии, где в качестве признаков используются степени исходной переменной $x$.

**Уравнение предсказания (гипотеза):**
$$\hat{y} = w_0 + w_1 x + w_2 x^2 + \dots + w_d x^d$$
где $d$ — степень полинома.

Матрица признаков (матрица Вандермонда) принимает вид:
$$X_{poly} = \begin{pmatrix} 1 & x_1 & x_1^2 & \dots & x_1^d \\ 1 & x_2 & x_2^2 & \dots & x_2^d \\ \dots & \dots & \dots & \dots & \dots \\ 1 & x_N & x_N^2 & \dots & x_N^d \end{pmatrix}$$

**Минимизируемая функция потерь (MSE - среднеквадратичная ошибка):**
$$L(\vec{w}) = \frac{1}{N} ||X_{poly}\vec{w} - \vec{y}||^2 \to \min_{\vec{w}}$$

### 1.1 Аналитическое решение
Для вычислительной устойчивости используется **псевдообратная матрица Мура-Пенроуза**:
$$\vec{w}^* = (X_{poly}^T X_{poly})^{+} X_{poly}^T \vec{y}$$

### 1.2 Градиентный спуск и нормализация (Z-score)
Перед шагом оптимизации мы применяем **Z-score стандартизацию** ко всем столбцам, кроме нулевого (столбца единиц):
$$X_{scaled}^{(j)} = \frac{X_{poly}^{(j)} - \mu_j}{\sigma_j}$$

### 1.3 Автоматический подбор оптимальной степени полинома
Чтобы алгоритм сам нашел лучшее значение $d$, применяется метод отложенной выборки:
1. Данные делятся на `Train` (обучающая, 80%) и `Test` (тестовая, 20%).
2. Модель обучается на `Train` для разных степеней $d \in [1, 2, \dots, 15]$.
3. Для каждой обученной модели измеряется ошибка MSE на `Test`.
4. Оптимальной считается та степень $d$, при которой ошибка на незнакомых данных (`Test MSE`) минимальна.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 7)

## 2. Реализация класса `MyPolynomialRegression`

In [ ]:
class MyPolynomialRegression:
    def __init__(self, degree=2, method='analytical', learning_rate=0.01, n_iters=1000):
        self.degree = degree
        self.method = method
        self.learning_rate = learning_rate
        self.n_iters = n_iters
        self.weights = None
        self.loss_history = []
        self.means = None
        self.stds = None
        
    def _transform_features(self, X):
        if X.ndim == 1:
            X = X.reshape(-1, 1)
            
        n_samples = X.shape[0]
        X_poly = np.ones((n_samples, 1))
        
        for d in range(1, self.degree + 1):
            X_poly = np.hstack((X_poly, X ** d))
            
        return X_poly

    def fit(self, X, y):
        X_poly = self._transform_features(X)
        n_samples, n_features = X_poly.shape
        
        if self.method == 'analytical':
            self.weights = np.linalg.pinv(X_poly.T @ X_poly) @ X_poly.T @ y
            
        elif self.method == 'gd':
            self.means = np.mean(X_poly[:, 1:], axis=0)
            self.stds = np.std(X_poly[:, 1:], axis=0)
            self.stds[self.stds == 0] = 1
            
            X_poly_scaled = np.copy(X_poly)
            X_poly_scaled[:, 1:] = (X_poly[:, 1:] - self.means) / self.stds
            
            self.weights = np.zeros(n_features)
            
            for i in range(self.n_iters):
                predictions = X_poly_scaled @ self.weights
                errors = predictions - y
                gradient = (2 / n_samples) * (X_poly_scaled.T @ errors)
                self.weights -= self.learning_rate * gradient
                mse = np.mean(errors ** 2)
                self.loss_history.append(mse)
        else:
            raise ValueError("Выберите method='analytical' или 'gd'")

    def predict(self, X):
        X_poly = self._transform_features(X)
        if self.method == 'gd':
            X_poly[:, 1:] = (X_poly[:, 1:] - self.means) / self.stds
        return X_poly @ self.weights
    
    @staticmethod
    def mse(y_true, y_pred):
        return np.mean((y_true - y_pred) ** 2)
    
    @staticmethod
    def r2_score(y_true, y_pred):
        ss_res = np.sum((y_true - y_pred) ** 2)
        ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
        return 1 - (ss_res / ss_tot)

## 3. Генерация данных и разделение на Train/Test
Скрытая функция: $y = 0.5x^3 - x^2 - 2x + 5 + \varepsilon$

In [ ]:
N_samples = 150
X = np.random.uniform(-3, 4, N_samples)
y_true = 0.5 * X**3 - X**2 - 2*X + 5
noise = np.random.normal(0, 3, size=N_samples)
y = y_true + noise

# Перемешиваем индексы и делим 80% на обучение, 20% на тест
indices = np.arange(N_samples)
np.random.shuffle(indices)

train_size = int(N_samples * 0.8)
X_train, y_train = X[indices[:train_size]], y[indices[:train_size]]
X_test, y_test = X[indices[train_size:]], y[indices[train_size:]]

print(f"Размер обучающей выборки: {X_train.shape[0]}")
print(f"Размер тестовой выборки: {X_test.shape[0]}")

X_plot = np.linspace(-3.5, 4.5, 200)

## 4. Автоматический подбор оптимальной степени (Кросс-валидация на отложенной выборке)
Пройдем циклом по степеням от 1 до 12. Найдем ту, где ошибка на тестовой выборке минимальна.

In [ ]:
max_degree = 12
degrees = list(range(1, max_degree + 1))
train_errors = []
test_errors = []

best_degree = 1
min_test_mse = float('inf')

for d in degrees:
    # Для скорости подбора используем аналитический метод
    model = MyPolynomialRegression(degree=d, method='analytical')
    model.fit(X_train, y_train)
    
    # Ошибки на Train и Test
    mse_train = model.mse(y_train, model.predict(X_train))
    mse_test = model.mse(y_test, model.predict(X_test))
    
    train_errors.append(mse_train)
    test_errors.append(mse_test)
    
    # Обновляем лучший результат
    if mse_test < min_test_mse:
        min_test_mse = mse_test
        best_degree = d

print(f"==> Оптимально подобранная степень полинома: {best_degree}")
print(f"==> Минимальная ошибка на тесте (MSE): {min_test_mse:.4f}\n")

# Построим график ошибок (логарифмическая шкала Y для наглядности)
plt.figure(figsize=(10, 5))
plt.plot(degrees, train_errors, label='Train MSE', marker='o', color='blue')
plt.plot(degrees, test_errors, label='Test MSE', marker='s', color='orange')
plt.axvline(best_degree, color='red', linestyle='--', label=f'Best degree = {best_degree}')
plt.title('График Train vs Test ошибки')
plt.xlabel('Степень полинома (d)')
plt.ylabel('MSE Loss')
plt.xticks(degrees)
plt.yscale('log')
plt.legend()
plt.show()

## 5. Обучение финальных моделей с оптимальной степенью
Теперь используем `best_degree` для обучения МНК и Градиентного спуска.

In [ ]:
# Градиентный спуск
model_gd = MyPolynomialRegression(degree=best_degree, method='gd', learning_rate=0.1, n_iters=500)
model_gd.fit(X_train, y_train)
y_pred_gd = model_gd.predict(X_test)

# Аналитическое решение
model_an = MyPolynomialRegression(degree=best_degree, method='analytical')
model_an.fit(X_train, y_train)
y_pred_an = model_an.predict(X_test)

print(f"=== Градиентный спуск (degree={best_degree}) ===")
print(f"Test MSE: {model_gd.mse(y_test, y_pred_gd):.4f}")
print(f"Test R^2: {model_gd.r2_score(y_test, y_pred_gd):.4f}\n")

print(f"=== Аналитическое решение (degree={best_degree}) ===")
print(f"Test MSE: {model_an.mse(y_test, y_pred_an):.4f}")
print(f"Test R^2: {model_an.r2_score(y_test, y_pred_an):.4f}")

# График предсказаний
plt.figure(figsize=(10, 6))
plt.scatter(X_train, y_train, color='gray', alpha=0.3, label='Train данные')
plt.scatter(X_test, y_test, color='blue', alpha=0.8, label='Test данные')

y_plot_gd = model_gd.predict(X_plot)
y_plot_an = model_an.predict(X_plot)

plt.plot(X_plot, y_plot_gd, color='crimson', linewidth=3, label='Градиентный спуск')
plt.plot(X_plot, y_plot_an, color='green', linestyle='--', linewidth=2, label='МНК (pinv)')

plt.title(f'Предсказания моделей с автовыбором (Degree={best_degree})')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.tight_layout()
plt.show()